# NLP Lab 4 Helper Notebook

This notebook contains completed code for the main TODO functions from the lab, along with short explanations of what each function does and how it works.

It is designed to be used alongside your original lab notebook. The code assumes that the earlier setup cells from the lab have already been run, including imports, dataset loading, tokenisation, `make_trainer`, `train`, `evaluate`, `pretrained_model`, and `finetuned_model`.


## Install and import libraries

If these libraries are not already installed in your environment, run the next cell first.
If you are using a managed university notebook environment where they are preinstalled, you can skip the install cell and go directly to the import cell.


In [10]:
# Uncomment and run this only if needed
#!pip install torch torchvision torchaudio transformers datasets evaluate


In [1]:
import torch
import torch.nn as nn

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

import evaluate


## 1. Count trainable parameters

This function counts how many scalar parameters will actually be updated during training.

- `model.parameters()` gives all parameter tensors.
- `requires_grad=True` means the parameter is trainable.
- `numel()` counts how many individual numbers are inside the tensor.


In [ ]:
def num_trainable_parameters(model):
    total = 0
    for param in model.parameters():
        if param.requires_grad:
            total += param.numel()
    return total


## 2. Head tuning

This function creates a model where only the final classification head is trainable.

In DistilBERT for sequence classification, the head is made of:
- `pre_classifier`
- `classifier`

Everything else is frozen.


In [17]:
from transformers import AutoModelForSequenceClassification

def make_headtuned_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )

    for param in model.parameters():
        param.requires_grad = False

    for param in model.pre_classifier.parameters():
        param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

    trained_model = train(model)
    return trained_model


## 3. Extract query and value layers

LoRA in this lab focuses on the query and value linear layers inside self-attention.

This function returns a dictionary mapping the full module name to the actual layer object.


In [6]:
def extract(model):
    named_layers = {}
    for i in range(6):
        q_name = f"distilbert.transformer.layer.{i}.attention.q_lin"
        v_name = f"distilbert.transformer.layer.{i}.attention.v_lin"
        named_layers[q_name] = model.get_submodule(q_name)
        named_layers[v_name] = model.get_submodule(v_name)
    return named_layers


## 4. Replace layers

This function takes a model and a dictionary of named layers, then injects each layer back into the right place in the model.

It is the inverse of `extract()`.


In [ ]:
def replace(model, named_layers):
    for full_name, new_layer in named_layers.items():
        parts = full_name.split(".")
        parent_path = ".".join(parts[:-1])
        child_name = parts[-1]
        parent_module = model.get_submodule(parent_path) if parent_path else model
        setattr(parent_module, child_name, new_layer)
    return model


## 5. Clone a linear layer

This helper creates a copy of a linear layer with the same weights and bias.

It is useful when you want a separate layer object but with the same values as the original.


In [ ]:
import torch
import torch.nn as nn

def clone_linear(original):
    out_features, in_features = original.weight.shape
    copy = nn.Linear(in_features, out_features)
    copy.load_state_dict(original.state_dict())
    return copy


## 6. Low-rank approximation with SVD

This function computes a rank-`r` approximation of a matrix using truncated singular value decomposition.

Idea:
- decompose the matrix into `U`, `S`, and `Vh`
- keep only the first `rank` singular values
- reconstruct an approximate matrix


In [ ]:
def approximate(matrix, rank):
    U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)
    U_r = U[:, :rank]
    S_r = S[:rank]
    Vh_r = Vh[:rank, :]
    return U_r @ torch.diag(S_r) @ Vh_r


## 7. Approximated model version 1

This version replaces each query/value layer in the head-tuned model with a low-rank approximation of the corresponding fully fine-tuned layer.

So here we approximate the full fine-tuned weight directly.


In [ ]:
def make_approximated_model_1(rank):
    model = AutoModelForSequenceClassification.from_pretrained("headtuned")
    ft_layers = extract(finetuned_model)
    base_layers = extract(model)

    new_layers = {}
    for name, ft_layer in ft_layers.items():
        base_layer = base_layers[name]
        new_layer = clone_linear(base_layer)
        with torch.no_grad():
            new_layer.weight.copy_(approximate(ft_layer.weight.data, rank))
            if ft_layer.bias is not None:
                new_layer.bias.copy_(ft_layer.bias.data)
        new_layers[name] = new_layer

    return replace(model, new_layers)


## 8. Approximated model version 2

This version follows the LoRA idea more closely.

Instead of approximating the full fine-tuned weight, it approximates only the update:

- `W0` = pretrained weight
- `W_ft` = fully fine-tuned weight
- `delta = W_ft - W0`
- approximate `delta`
- rebuild weight as `W0 + approximated_delta`


In [ ]:
def make_approximated_model_2(rank):
    model = AutoModelForSequenceClassification.from_pretrained("headtuned")

    pretrained_layers = extract(pretrained_model)
    finetuned_layers = extract(finetuned_model)
    base_layers = extract(model)

    new_layers = {}
    for name in finetuned_layers:
        W0 = pretrained_layers[name].weight.data
        W_ft = finetuned_layers[name].weight.data
        delta = W_ft - W0
        delta_r = approximate(delta, rank)
        W_new = W0 + delta_r

        new_layer = clone_linear(base_layers[name])
        with torch.no_grad():
            new_layer.weight.copy_(W_new)
            if finetuned_layers[name].bias is not None:
                new_layer.bias.copy_(finetuned_layers[name].bias.data)
        new_layers[name] = new_layer

    return replace(model, new_layers)


## 9. LoRA adapter class

This class wraps a pretrained linear layer.

It computes:

`output = pretrained(x) + scaling * B(A(x))`

Where:
- the pretrained layer is frozen
- `A` maps input to a low-rank space
- `B` maps back to output size
- scaling is `alpha / rank`

The initialization follows the lab instructions:
- `A` is random normal
- `B` is zero


In [ ]:
class LoRA(nn.Module):
    def __init__(self, pretrained, rank=12, alpha=24):
        super().__init__()
        self.pretrained = pretrained
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        in_features = pretrained.in_features
        out_features = pretrained.out_features

        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)

        nn.init.normal_(self.A.weight, mean=0.0, std=1.0)
        nn.init.zeros_(self.B.weight)

        for param in self.pretrained.parameters():
            param.requires_grad = False

    def forward(self, x):
        base_out = self.pretrained(x)
        update = self.B(self.A(x)) * self.scaling
        return base_out + update


## 10. Create a LoRA model

This function injects LoRA into every query and value layer.

Then it:
- freezes the full model
- unfreezes LoRA parameters
- unfreezes the classification head
- trains the model

The assignment says to use `alpha = 2 * rank`.


In [ ]:
def make_lora_model(rank):
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )

    lora_layers = {}
    for i in range(6):
        q_name = f"distilbert.transformer.layer.{i}.attention.q_lin"
        v_name = f"distilbert.transformer.layer.{i}.attention.v_lin"

        q_layer = model.get_submodule(q_name)
        v_layer = model.get_submodule(v_name)

        lora_layers[q_name] = LoRA(q_layer, rank=rank, alpha=2 * rank)
        lora_layers[v_name] = LoRA(v_layer, rank=rank, alpha=2 * rank)

    model = replace(model, lora_layers)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.pre_classifier.parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True

    for module in model.modules():
        if isinstance(module, LoRA):
            for param in module.parameters():
                param.requires_grad = True

    trained_model = train(model)
    return trained_model


## 11. Suggested quick checks

After pasting or running the functions, you can test them using checks like these.


In [5]:
# Task 4.01
num_trainable_parameters(pretrained_model)

# Task 4.02
headtuned_model = make_headtuned_model()
num_trainable_parameters(headtuned_model)

# Task 4.03
extracted = extract(pretrained_model)
extracted.keys()

# Task 4.05
original = torch.rand(768, 8) @ torch.rand(8, 384)
approximation = approximate(original, 8)
torch.dist(original, approximation)

# Task 4.09
lora_model = make_lora_model(6)
num_trainable_parameters(lora_model)
evaluate(lora_model)


NameError: name 'pretrained_model' is not defined